# LouisFarm — Semaine 5 : SQL & Data Extraction
## Base de donnees : MFB Benin (SQLite)

**Objectif :** Maitriser l'extraction depuis des bases relationnelles.

In [ ]:
import sqlite3, pandas as pd, matplotlib.pyplot as plt
import sys; sys.path.insert(0, ".")
from utils_louisfarm import gen_mfb_benin_sqlite
import warnings; warnings.filterwarnings("ignore")

try:
    gen_mfb_benin_sqlite()
    print("Base SQLite generee")
except Exception as e:
    print(f"Base existante: {e}")

conn = sqlite3.connect("./mfb_benin.sqlite")
print("Connexion OK")

# Structure
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
for t in tables["name"]:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", conn).iloc[0,0]
    cols = pd.read_sql(f"PRAGMA table_info({t})", conn)["name"].tolist()
    print(f"  {t:<15}: {count:>6} lignes | {cols}")

## Lecon 5.2 — Jointures SQL

In [ ]:
# INNER JOIN: Clients + Comptes
q1 = pd.read_sql(
    "SELECT c.nom, c.prenom, c.region, co.type, co.solde_actuel, co.statut "
    "FROM clients c INNER JOIN comptes co ON c.id = co.client_id "
    "ORDER BY co.solde_actuel DESC LIMIT 8",
    conn)
print("JOIN Clients-Comptes:")
print(q1.to_string(index=False))

# GROUP BY avec jointure
q2 = pd.read_sql(
    "SELECT c.region, COUNT(DISTINCT c.id) as clients, "
    "SUM(co.solde_actuel) as encours, AVG(co.solde_actuel) as solde_moy "
    "FROM clients c JOIN comptes co ON c.id = co.client_id "
    "WHERE co.statut = 'Actif' GROUP BY c.region ORDER BY encours DESC",
    conn)
print("\nEncours par region:")
print(q2.round(0).to_string(index=False))

## Lecon 5.3 — CTE et Window Functions

In [ ]:
# CTE (WITH ... AS)
q_cte = pd.read_sql(
    "WITH tx_par_compte AS ("
    "  SELECT compte_id, COUNT(*) as nbr_tx, SUM(montant) as total "
    "  FROM transactions GROUP BY compte_id"
    "), enrichi AS ("
    "  SELECT c.nom, c.region, co.type, tx.nbr_tx, tx.total "
    "  FROM clients c JOIN comptes co ON c.id=co.client_id "
    "  JOIN tx_par_compte tx ON co.id=tx.compte_id"
    ") SELECT * FROM enrichi ORDER BY total DESC LIMIT 10",
    conn)
print("TOP 10 par transactions (CTE):")
print(q_cte.to_string(index=False))

# Window Function
q_window = pd.read_sql(
    "SELECT type_tx, COUNT(*) as nbr, SUM(montant) as total_montant, "
    "AVG(montant) as moy FROM transactions GROUP BY type_tx ORDER BY total_montant DESC",
    conn)
print("\nActivite transactionnelle:")
print(q_window.round(0).to_string(index=False))

conn.close()
print("\nConnexion fermee")

## Lecon 5.4 — Pipeline SQL → Python → Visualisation

In [ ]:
conn = sqlite3.connect("./mfb_benin.sqlite")
df_sql = pd.read_sql(
    "SELECT c.region, c.genre, AVG(co.solde_actuel) as solde_moyen, "
    "COUNT(DISTINCT c.id) as nbr_clients FROM clients c "
    "JOIN comptes co ON c.id=co.client_id WHERE co.statut='Actif' "
    "GROUP BY c.region, c.genre",
    conn)
conn.close()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Semaine 5 - SQL vers Visualisation : MFB Benin", fontweight="bold")

by_reg = df_sql.groupby("region")["solde_moyen"].mean().sort_values()
axes[0].barh(by_reg.index, by_reg.values/1000, color="#2E86AB")
axes[0].set_xlabel("Solde moyen (milliers FCFA)")
axes[0].set_title("Solde moyen actif par region")

by_genre = df_sql.groupby("genre")["solde_moyen"].mean()
axes[1].bar(by_genre.index, by_genre.values/1000, color=["#2E86AB","#A23B72"])
axes[1].set_title("Solde moyen par genre")
axes[1].set_ylabel("Milliers FCFA")

plt.tight_layout()
plt.savefig("./s5_sql_viz.png", dpi=100, bbox_inches="tight")
plt.show()
print("Sauvegarde: s5_sql_viz.png")